# Pillar A — decision-quantity calibration (IC-8 `DerivedQuantity`)

This notebook reproduces workstream A's headline check end to end: **A5 — Decision-quantity UQ (IC-8
surface)**. A5's Definition of Done asks that a Monte-Carlo decision quantity (`prob_exceed`,
`tonnage_above_cutoff`, `net_pay`, `drill_target_prob`) report a 90% credible interval whose *empirical*
coverage of the truth-evaluated value, across many synthetic realizations, is at least 0.85.

We reproduce that check for `prob_exceed`: a 1-D field with a region of interest, a per-cell conjugate
Gaussian posterior, run across 300 independent realizations committed to
`data/pillar_validation/a_decision_quantities.npz` (each realization's true field and noisy observation,
generated once with a fixed seed).

**Judgment call.** `mixle_pde.decision_quantities` (A5) and `mixle.reason.posterior_protocol` (IC-1) had
not landed on `release/0.8.0` at the time this notebook was written (both still raise `NotImplementedError`
Wave-0 stubs, or don't exist yet, in the sibling checkouts). We import them if present and otherwise fall
back to a small reference implementation of the exact A5 DR-ALG (Monte Carlo over posterior draws), so this
notebook is reproducible today and will pick up the real modules transparently the moment they merge.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

try:
    from mixle.reason.posterior_protocol import Posterior  # IC-1
    from mixle_pde.decision_quantities import prob_exceed as _real_prob_exceed  # A5
    HAVE_REAL_A5 = True
except ImportError:
    HAVE_REAL_A5 = False

print("using landed mixle_pde.decision_quantities / IC-1 Posterior:", HAVE_REAL_A5)

## 1. Load the committed fixture

`x` is the 1-D cell-center grid, `truths[r]`/`obs[r]` are the true field and its noisy observation for realization `r` (300 realizations, generated once with `np.random.default_rng(0)`).

In [ ]:
data = np.load("../../../data/pillar_validation/a_decision_quantities.npz")
x, truths, obs, obs_var = data["x"], data["truths"], data["obs"], float(data["obs_var"])
C = x.size
region = x > 0.5          # the region of interest A5's `prob_exceed` is evaluated over
threshold = 0.55
print(f"C={C} cells, {truths.shape[0]} realizations, region has {region.sum()} cells")

## 2. A per-cell conjugate-Gaussian posterior (this notebook's IC-1 `Posterior`)

Each cell has an independent Gaussian prior `N(0.5, prior_var)`; combined with a Gaussian observation of known `obs_var`, the posterior is the standard conjugate update. This satisfies the IC-1 `Posterior` shape (`samples`, `mean`, `cov`, `credible_interval`, `derived_quantity`) by construction.

In [ ]:
from dataclasses import dataclass, field

PRIOR_MEAN, PRIOR_VAR = 0.5, 0.07 ** 2


@dataclass
class DerivedQuantity:
    """IC-1 `DerivedQuantity`: samples of a pushforward + the honesty flag."""
    samples: np.ndarray
    prior_dominated: bool = False

    def credible_interval(self, level: float):
        a = (1.0 - level) / 2.0
        return np.quantile(self.samples, a), np.quantile(self.samples, 1 - a)


class GaussianFieldPosterior:
    """IC-1 `Posterior` over a per-cell independent Gaussian field."""

    def __init__(self, post_mean, post_var):
        self.post_mean = post_mean
        self.post_var = post_var

    def samples(self, n, rng):
        return self.post_mean[None, :] + np.sqrt(self.post_var) * rng.standard_normal((n, self.post_mean.size))

    @property
    def mean(self):
        return self.post_mean

    @property
    def cov(self):
        return np.diag(np.full_like(self.post_mean, self.post_var))

    def credible_interval(self, level):
        z = {0.9: 1.6448536269514722}.get(level) or __import__("scipy.stats", fromlist=["norm"]).norm.ppf(0.5 + level / 2)
        sd = np.sqrt(self.post_var)
        return self.post_mean - z * sd, self.post_mean + z * sd

    def derived_quantity(self, fn, n, rng):
        return DerivedQuantity(samples=fn(self.samples(n, rng)))


def fallback_prob_exceed(posterior, region, *, threshold, n=4096, rng=None):
    """A5 `prob_exceed(posterior, region, *, threshold, n=4096, rng=None) -> DerivedQuantity`:
    the posterior distribution of the region fraction where `field > threshold`."""
    rng = rng or np.random.default_rng()
    draws = posterior.samples(n, rng)
    frac = (draws[:, region] > threshold).mean(axis=1)
    return DerivedQuantity(samples=frac, prior_dominated=False)


prob_exceed = _real_prob_exceed if HAVE_REAL_A5 else fallback_prob_exceed

## 3. Monte-Carlo calibration sweep across 300 realizations

For each committed realization, fit the conjugate posterior from the fixture's observation, draw the `prob_exceed` `DerivedQuantity`, and check whether its 90% credible interval covers the true, truth-evaluated fraction. The empirical coverage across all realizations is A5's calibration number.

In [ ]:
post_rng = np.random.default_rng(123)  # fixed seed for the posterior Monte-Carlo draws (determinism)
n_draws = 2000
coverage_flags = []
pit_values = []

for r in range(truths.shape[0]):
    truth, o = truths[r], obs[r]
    post_var = 1.0 / (1.0 / PRIOR_VAR + 1.0 / obs_var)
    post_mean = post_var * (PRIOR_MEAN / PRIOR_VAR + o / obs_var)
    posterior = GaussianFieldPosterior(post_mean, post_var)

    dq = prob_exceed(posterior, region, threshold=threshold, n=n_draws, rng=post_rng)
    lo, hi = dq.credible_interval(0.9)
    true_frac = (truth[region] > threshold).mean()

    coverage_flags.append(lo <= true_frac <= hi)
    pit_values.append((dq.samples <= true_frac).mean())  # probability-integral-transform rank

coverage = float(np.mean(coverage_flags))
print(f"empirical 90% CI coverage over {len(coverage_flags)} realizations: {coverage:.3f}")

## 4. Calibration diagnostic: the PIT histogram

If the posterior is well calibrated, the rank of the truth-evaluated `prob_exceed` value within its own posterior draws (the probability-integral transform) is uniform on `[0, 1]`. A flat histogram is the visual signature of calibrated UQ -- this is the plot A5's exec DoD only asserts a number for.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].hist(pit_values, bins=20, range=(0, 1), color="#2E86AB", edgecolor="white")
ax[0].axhline(len(pit_values) / 20, color="#444", ls="--", lw=1, label="uniform reference")
ax[0].set_title("PIT histogram (prob_exceed)"); ax[0].set_xlabel("PIT rank"); ax[0].legend()

flags = np.array(coverage_flags, dtype=float)
running = np.cumsum(flags) / (np.arange(len(flags)) + 1)
ax[1].plot(running, color="#A23B72")
ax[1].axhline(0.85, color="#C0392B", ls="--", label="A5 threshold (0.85)")
ax[1].axhline(0.9, color="#444", ls=":", lw=1, label="nominal (0.90)")
ax[1].set_title("running empirical coverage"); ax[1].set_xlabel("realization"); ax[1].legend()
plt.tight_layout(); plt.show()

## 5. Definition of Done

Reproduces A5's exec-DoD threshold: empirical 90% CI coverage of `prob_exceed` across independent synthetic realizations must be `>= 0.85`.

In [ ]:
assert coverage >= 0.85, f"A5 calibration failed: coverage {coverage:.3f} < 0.85"
print(f"PASS -- prob_exceed 90% CI empirical coverage = {coverage:.3f} >= 0.85")